In [4]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline 
from pyannote.audio import Pipeline 
from dotenv import load_dotenv

load_dotenv()
HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
os.environ["PATH"] += os.pathsep + r"D:\ksol927_Study_Ai\ffmpeg-2026-08-20-git-7d77562d2a-full_build\bin"

# pyannote checkpoints are trusted Hugging Face model files.
_original_torch_load = torch.load

def _trusted_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)


torch.load = _trusted_torch_load


def whisper_stt(
    audio_file_path: str,
    output_file_path: str = "output.csv"
):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_id = "openai/whisper-large-v3-turbo"

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        return_timestamps=True,
        chunk_length_s=10,
        stride_length_s=2,
    )

    result = pipe(audio_file_path)
    df = whisper_to_dataframe(result, output_file_path)

    return result, df


def whisper_to_dataframe(result, output_file_path):
    start_end_text = []

    for chunk in result["chunks"]:
        start = chunk["timestamp"][0]
        end = chunk["timestamp"][1]
        text = chunk["text"].strip()
        start_end_text.append([start, end, text])
        df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
        df.to_csv(output_file_path, index=False, sep="|")

    return df


def speaker_diarization(
        audio_file_path: str,
        output_rttm_file_path: str,
        output_csv_file_path: str
    ):
    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=HUGGING_FACE_TOKEN
    )

    if torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))
        print("cuda is available")
    else:
        print("cuda is not available")
    diarization_pipeline = pipeline(audio_file_path)

    with open(output_rttm_file_path, "w", encoding="utf-8") as rttm:
        diarization_pipeline.write_rttm(rttm)

    df_rttm = pd.read_csv(
        output_rttm_file_path,
        sep=" ",
        header=None,
        names=["type", "file", "chnl", "start", "duration", "C1", "C2", "speaker_id", "C3", "C4"]
    )

    df_rttm["end"] = df_rttm["start"] + df_rttm["duration"]
    df_rttm["number"] = None
    df_rttm.at[0, "number"] = 0

    for i in range(1, len(df_rttm)):
        if df_rttm.at[i, "speaker_id"] != df_rttm.at[i - 1, "speaker_id"]:
            df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"] + 1
        else:
            df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"]

    df_rttm_grouped = df_rttm.groupby("number").agg(
        start=pd.NamedAgg(column="start", aggfunc="min"),
        end=pd.NamedAgg(column="end", aggfunc="max"),
        speaker_id=pd.NamedAgg(column="speaker_id", aggfunc="first")
    )

    df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
    df_rttm_grouped.to_csv(output_csv_file_path, index=False, encoding="utf-8")
    return df_rttm_grouped


def stt_to_rttm(
        audio_file_path: str,
        stt_output_file_path: str,
        rttm_file_path: str,
        rttm_csv_file_path: str,
        final_output_csv_file_path: str
    ):
    result, df_stt = whisper_stt(audio_file_path, stt_output_file_path)
    df_rttm = speaker_diarization(audio_file_path, rttm_file_path, rttm_csv_file_path)
    df_rttm["text"] = ""

    for _, row_stt in df_stt.iterrows():
        overlap_dict = {}
        for i_rttm, row_rttm in df_rttm.iterrows():
            overlap = max(0, min(row_stt["end"], row_rttm["end"]) - max(row_stt["start"], row_rttm["start"]))
            overlap_dict[i_rttm] = overlap

        max_overlap = max(overlap_dict.values())
        max_overlap_idx = max(overlap_dict, key=overlap_dict.get)
        if max_overlap > 0:
            df_rttm.at[max_overlap_idx, "text"] += row_stt["text"] + "\n"

    df_rttm.to_csv(
        final_output_csv_file_path,
        index=False,
        sep="|",
        encoding="utf-8"
    )
    return df_rttm


if __name__ == "__main__":
    audio_file_path = "audio/싼기타_비싼기타.mp3"
    stt_output_file_path = "output/싼기타_비싼기타.csv"
    rttm_file_path = "output/싼기타_비싼기타.rttm"
    rttm_csv_file_path = "output/싼기타_비싼기타_rttm.csv"
    final_csv_file_path = "output/싼기타_비싼기타_final.csv"

    df_rttm = stt_to_rttm(
        audio_file_path,
        stt_output_file_path,
        rttm_file_path,
        rttm_csv_file_path,
        final_csv_file_path
    )

    print(df_rttm)

c:\Users\ksol9\Documents\ksol927_ai_study\STUDY\ksol927\chap05\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
c:\Users\ksol9\Documents\ksol927_ai_study\STUDY\ksol927\chap05\.venv\Lib\site-packages\pyannote\audio\core\io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


cuda is not available


c:\Users\ksol9\Documents\ksol927_ai_study\STUDY\ksol927\chap05\.venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
c:\Users\ksol9\Documents\ksol927_ai_study\STUDY\ksol927\chap05\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, cor

          start      end  speaker_id  duration  \
number                                           
0         0.993   30.204  SPEAKER_01    29.211   
1        32.414   42.708  SPEAKER_00    10.294   
2        41.645   44.024  SPEAKER_01     2.379   
3        45.813   67.109  SPEAKER_00    21.296   
4        67.227   82.786  SPEAKER_01    15.559   
5        84.659  102.564  SPEAKER_00    17.905   
6       103.492  117.532  SPEAKER_01    14.040   
7       119.759  138.676  SPEAKER_00    18.917   
8       139.351  168.967  SPEAKER_01    29.616   
9       170.907  192.321  SPEAKER_00    21.414   
10      192.322  193.689  SPEAKER_01     1.367   
11      192.760  193.503  SPEAKER_00     0.743   
12      193.823  216.571  SPEAKER_01    22.748   
13      218.579  238.120  SPEAKER_00    19.541   
14      238.120  238.677  SPEAKER_01     0.557   
15      238.188  239.352  SPEAKER_00     1.164   
16      239.858  240.651  SPEAKER_01     0.793   
17      240.297  241.006  SPEAKER_00     0.709   
